In [4]:
# # Clear output folder
# import os

# def remove_folder_contents(folder):
#     for the_file in os.listdir(folder):
#         file_path = os.path.join(folder, the_file)
#         try:
#             if os.path.isfile(file_path):
#                 os.unlink(file_path)
#             elif os.path.isdir(file_path):
#                 remove_folder_contents(file_path)
#                 os.rmdir(file_path)
#         except Exception as e:
#             print(e)

# folder_path = '/kaggle/working'
# remove_folder_contents(folder_path)
# # os.rmdir(folder_path)

In [1]:
import io
import bisect
import gc
import zipfile
from pathlib import Path

from PIL import Image
from tqdm.notebook import tqdm

## Paths and Setting

In [5]:
# /kaggle/input/competitions/diabetic-retinopathy-detection
INPUT_DIR = Path("/kaggle/input/competitions/diabetic-retinopathy-detection")
WORK_DIR = Path("/kaggle/working")

OUTPUT_DIR = WORK_DIR / "processed_chunks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = 300
JPEG_QUALITY = 80
BATCH_SIZE = 2000   # images per output zip
BASE_NAME = "train"

## Checking Files

In [ ]:
print("Files in dataset directory:")
for f in sorted(WORK_DIR.iterdir()):
    print(f.name)

## Multipart reader for split zip files

In [6]:
class MultiPartFile(io.RawIOBase):
    def __init__(self, part_paths):
        self.part_paths = [Path(p) for p in part_paths]
        self.part_sizes = [p.stat().st_size for p in self.part_paths]

        self.part_starts = []
        total = 0
        for size in self.part_sizes:
            self.part_starts.append(total)
            total += size

        self.total_size = total
        self.pos = 0
        self.current_idx = None
        self.current_fh = None
        super().__init__()

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def _locate(self, absolute_pos):
        if absolute_pos < 0 or absolute_pos > self.total_size:
            raise ValueError("Position out of range")

        if absolute_pos == self.total_size:
            return len(self.part_paths) - 1, self.part_sizes[-1]

        idx = bisect.bisect_right(self.part_starts, absolute_pos) - 1
        local_pos = absolute_pos - self.part_starts[idx]
        return idx, local_pos

    def _open_part(self, idx, local_pos):
        if self.current_idx != idx:
            if self.current_fh:
                self.current_fh.close()
            self.current_fh = open(self.part_paths[idx], "rb")
            self.current_idx = idx
        self.current_fh.seek(local_pos)

    def read(self, n=-1):
        if self.pos >= self.total_size:
            return b""

        if n is None or n < 0:
            n = self.total_size - self.pos

        remaining = min(n, self.total_size - self.pos)
        chunks = []

        while remaining > 0 and self.pos < self.total_size:
            idx, local_pos = self._locate(self.pos)
            self._open_part(idx, local_pos)

            bytes_left_in_part = self.part_sizes[idx] - local_pos
            to_read = min(remaining, bytes_left_in_part)

            data = self.current_fh.read(to_read)
            if not data:
                break

            chunks.append(data)
            got = len(data)
            self.pos += got
            remaining -= got

        return b"".join(chunks)

    def seek(self, offset, whence=io.SEEK_SET):
        if whence == io.SEEK_SET:
            new_pos = offset
        elif whence == io.SEEK_CUR:
            new_pos = self.pos + offset
        elif whence == io.SEEK_END:
            new_pos = self.total_size + offset
        else:
            raise ValueError("Invalid whence")

        if new_pos < 0:
            raise ValueError("Negative seek position")

        self.pos = min(new_pos, self.total_size)
        return self.pos

    def close(self):
        if self.current_fh:
            self.current_fh.close()
            self.current_fh = None
        self.current_idx = None
        super().close()

## Helper Function

In [9]:
def get_split_parts(input_dir, base_name="train"):
    part_files = sorted(input_dir.glob(f"{base_name}.zip.*"))
    if not part_files:
        raise FileNotFoundError(f"No files found for {base_name}.zip.*")
    return part_files


def preprocess_image(file_obj, target_size=384, jpeg_quality=75):
    img = Image.open(file_obj).convert("RGB")
    img = img.resize((target_size, target_size), Image.Resampling.LANCZOS)

    out_buffer = io.BytesIO()
    img.save(out_buffer, format="JPEG", quality=jpeg_quality, optimize=True)
    return out_buffer.getvalue()

In [7]:
def process_split_zip_to_chunks(input_dir, output_dir, base_name="train",
                                target_size=384, jpeg_quality=75, batch_size=2000):
    part_files = get_split_parts(input_dir, base_name)

    chunk_paths = []

    with MultiPartFile(part_files) as mpf:
        with zipfile.ZipFile(mpf, "r") as in_zip:
            image_infos = [
                info for info in in_zip.infolist()
                if not info.is_dir() and info.filename.lower().endswith((".jpg", ".jpeg", ".png"))
            ]

            print(f"Total images found inside {base_name}: {len(image_infos)}")

            chunk_idx = 1
            for start in range(0, len(image_infos), batch_size):
                end = min(start + batch_size, len(image_infos))
                batch = image_infos[start:end]

                chunk_zip_path = output_dir / f"{base_name}_processed_part_{chunk_idx:03d}.zip"
                print(f"\nCreating {chunk_zip_path.name} with images {start} to {end - 1}")

                with zipfile.ZipFile(chunk_zip_path, "w", compression=zipfile.ZIP_STORED) as out_zip:
                    for info in tqdm(batch):
                        try:
                            with in_zip.open(info) as src:
                                processed_bytes = preprocess_image(
                                    src,
                                    target_size=target_size,
                                    jpeg_quality=jpeg_quality
                                )

                            out_name = f"{Path(info.filename).stem}.jpg"
                            out_zip.writestr(out_name, processed_bytes)

                        except Exception as e:
                            print(f"Failed: {info.filename} -> {e}")

                chunk_paths.append(chunk_zip_path)
                chunk_idx += 1
                gc.collect()

    return chunk_paths

In [10]:
chunk_paths = process_split_zip_to_chunks(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    base_name=BASE_NAME,
    target_size=TARGET_SIZE,
    jpeg_quality=JPEG_QUALITY,
    batch_size=BATCH_SIZE
)

print("\nCreated chunk files:")
for p in chunk_paths:
    print(p)

Total images found inside train: 35126

Creating train_processed_part_001.zip with images 0 to 1999


  0%|          | 0/2000 [00:00<?, ?it/s]


Creating train_processed_part_002.zip with images 2000 to 3999


  0%|          | 0/2000 [00:00<?, ?it/s]


Creating train_processed_part_003.zip with images 4000 to 5999


...


Creating train_processed_part_017.zip with images 32000 to 33999


  0%|          | 0/2000 [00:00<?, ?it/s]


Creating train_processed_part_018.zip with images 34000 to 35125


  0%|          | 0/1126 [00:00<?, ?it/s]


Created chunk files:
/kaggle/working/processed_chunks/train_processed_part_001.zip
/kaggle/working/processed_chunks/train_processed_part_002.zip
/kaggle/working/processed_chunks/train_processed_part_003.zip
/kaggle/working/processed_chunks/train_processed_part_004.zip
/kaggle/working/processed_chunks/train_processed_part_005.zip
/kaggle/working/processed_chunks/train_processed_part_006.zip
/kaggle/working/processed_chunks/train_processed_part_007.zip
/kaggle/working/processed_chunks/train_processed_part_008.zip
/kaggle/working/processed_chunks/train_processed_part_009.zip
/kaggle/working/processed_chunks/train_processed_part_010.zip
/kaggle/working/processed_chunks/train_processed_part_011.zip
/kaggle/working/processed_chunks/train_processed_part_012.zip
/kaggle/working/processed_chunks/train_processed_part_013.zip
/kaggle/working/processed_chunks/train_processed_part_014.zip
/kaggle/working/processed_chunks/train_processed_part_015.zip
/kaggle/working/processed_chunks/train_processed

In [11]:
for p in chunk_paths:
    size_gb = p.stat().st_size / (1024 ** 3)
    print(f"{p.name}: {size_gb:.2f} GB")

train_processed_part_001.zip: 0.01 GB
train_processed_part_002.zip: 0.01 GB
train_processed_part_003.zip: 0.01 GB
train_processed_part_004.zip: 0.01 GB
train_processed_part_005.zip: 0.01 GB
train_processed_part_006.zip: 0.02 GB
train_processed_part_007.zip: 0.01 GB
train_processed_part_008.zip: 0.01 GB
train_processed_part_009.zip: 0.01 GB
train_processed_part_010.zip: 0.01 GB
train_processed_part_011.zip: 0.01 GB
train_processed_part_012.zip: 0.02 GB
train_processed_part_013.zip: 0.01 GB
train_processed_part_014.zip: 0.01 GB
train_processed_part_015.zip: 0.01 GB
train_processed_part_016.zip: 0.01 GB
train_processed_part_017.zip: 0.01 GB
train_processed_part_018.zip: 0.01 GB


In [12]:
LABELS_DIR = WORK_DIR / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_DIR / "trainLabels.csv.zip", "r") as zf:
    zf.extractall(LABELS_DIR)

print("Extracted label files:")
for f in LABELS_DIR.iterdir():
    print(f.name)

Extracted label files:
trainLabels.csv


In [13]:
import zipfile
from pathlib import Path
from tqdm.notebook import tqdm

WORK_DIR = Path("/kaggle/working")
CHUNKS_DIR = WORK_DIR / "processed_chunks"
FINAL_ZIP = WORK_DIR / "train_processed_final.zip"

chunk_files = sorted(CHUNKS_DIR.glob("train_processed_part_*.zip"))
print("Chunks found:", len(chunk_files))

with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_STORED) as final_zip:
    for chunk_zip in tqdm(chunk_files):
        with zipfile.ZipFile(chunk_zip, "r") as zf:
            for name in zf.namelist():
                final_zip.writestr(name, zf.read(name))

print("Final merged zip created:", FINAL_ZIP)
print("Final size (GB):", round(FINAL_ZIP.stat().st_size / (1024**3), 2))

Chunks found: 18


  0%|          | 0/18 [00:00<?, ?it/s]

Final merged zip created: /kaggle/working/train_processed_final.zip
Final size (GB): 0.26


In [14]:
print(FINAL_ZIP.exists(), FINAL_ZIP)
print("Size in GB:", round(FINAL_ZIP.stat().st_size / (1024**3), 2))

True /kaggle/working/train_processed_final.zip
Size in GB: 0.26


In [15]:
for p in chunk_files:
    p.unlink()

print("Deleted chunk zips.")

Deleted chunk zips.
